# A/B Testing for Agents and RAG Systems

This notebook demonstrates the most popular A/B testing scenarios for:

1. **Agent Systems**: Model comparison, prompt engineering, tool selection

2. **RAG Systems**: Chunk size, retrieval methods, reranking strategies

## Table of Contents

- [Setup](#setup)

- [Agent A/B Testing](#agent-ab-testing)

  - [Model Comparison](#model-comparison)

  - [Prompt Engineering](#prompt-engineering)

  - [Temperature Tuning](#temperature-tuning)

- [RAG A/B Testing](#rag-ab-testing)

  - [Chunk Size Optimization](#chunk-size-optimization)

  - [Retrieval Strategy](#retrieval-strategy)

  - [Reranking Comparison](#reranking-comparison)

  - [Embedding Model Selection](#embedding-model-selection)

- [Results Analysis](#results-analysis)

## Setup

In [ ]:
from dotenv import load_dotenv
import os

# Load environment variables
load_dotenv(dotenv_path='../.env')

base_url = ""
api_key = os.environ['UNIFIED_LLM_KEY']

# Set LangSmith project
os.environ["LANGSMITH_PROJECT"] = "AB_Testing"

# Verify LangSmith API key
if os.environ.get('LANGSMITH_API_KEY'):
    print("✅ LangSmith API key loaded")
else:
    print("⚠️  WARNING: LANGSMITH_API_KEY not found!")

# Agent A/B Testing

## Popular Agent A/B Tests:

| Test Type | Variant A | Variant B | What to Measure |

|-----------|-----------|-----------|------------------|

| **Model Comparison** | GPT-4o | GPT-4o-mini | Accuracy, Cost, Latency |

| **Prompt Engineering** | Detailed instructions | Concise instructions | Task completion rate |

| **Temperature** | temperature=0 | temperature=0.3 | Creativity vs Consistency |

| **Tool Selection** | All tools | Curated subset | Precision, Efficiency |

| **Agent Framework** | ReAct | Plan-and-Execute | Success rate, Steps |

| **Context Window** | Full context | Summarized context | Quality vs Cost |

## Test 1: Model Comparison (GPT-4o vs GPT-4o-mini)

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langsmith import traceable

# Create sample task
agent_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI assistant that solves math word problems step by step."),
    ("user", "{question}")
])

# Variant A: GPT-4o (More capable, expensive)
@traceable(name="agent_gpt4o")
def agent_variant_a(question: str) -> dict:
    """Agent using GPT-4o model"""
    llm = ChatOpenAI(model="gpt-4o", temperature=0, base_url=base_url, api_key=api_key)
    chain = agent_prompt | llm | StrOutputParser()
    answer = chain.invoke({"question": question})
    return {"answer": answer, "model": "gpt-4o"}

# Variant B: GPT-4o-mini (Faster, cheaper)
@traceable(name="agent_gpt4o_mini")
def agent_variant_b(question: str) -> dict:
    """Agent using GPT-4o-mini model"""
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, base_url=base_url, api_key=api_key)
    chain = agent_prompt | llm | StrOutputParser()
    answer = chain.invoke({"question": question})
    return {"answer": answer, "model": "gpt-4o-mini"}

print("✅ Model comparison agents defined")

In [ ]:
# Create evaluation dataset
from langsmith import Client

client = Client()
dataset_name = "Math_Word_Problems"

examples = [
    ("A store has 50 apples. They sell 30% of them. How many apples are left?", "35 apples"),
    ("If a car travels 60 mph for 2.5 hours, how far does it go?", "150 miles"),
    ("A pizza is cut into 8 slices. If you eat 3/8 of the pizza, how many slices did you eat?", "3 slices"),
]

try:
    dataset = client.read_dataset(dataset_name=dataset_name)
    print(f"✅ Using existing dataset: {dataset_name}")
except:
    dataset = client.create_dataset(dataset_name=dataset_name, description="Math word problems for agent evaluation")
    inputs_list = [{"question": q} for q, _ in examples]
    outputs_list = [{"answer": a} for _, a in examples]
    client.create_examples(inputs=inputs_list, outputs=outputs_list, dataset_id=dataset.id)
    print(f"✅ Created dataset with {len(examples)} examples")

In [ ]:
# Prediction functions for evaluation
def predict_variant_a(example: dict) -> dict:
    return agent_variant_a(example["question"])

def predict_variant_b(example: dict) -> dict:
    return agent_variant_b(example["question"])

In [ ]:
# Custom evaluators for accuracy
from langsmith.schemas import Run, Example
from langchain_openai import ChatOpenAI

# Method 1: Simple accuracy evaluator (no LLM needed)
def accuracy_evaluator(run: Run, example: Example) -> dict:
    """Evaluate if answer contains the expected result"""
    prediction = run.outputs.get("answer", "").lower()
    reference = example.outputs.get("answer", "").lower()

    # Check if key numbers/words from reference are in prediction
    score = 1.0 if reference in prediction else 0.0

    return {"key": "accuracy", "score": score}

# Method 2: LLM-as-Judge evaluator (with custom LLM)
def llm_judge_evaluator(run: Run, example: Example) -> dict:
    """Use LLM to judge answer quality"""
    prediction = run.outputs.get("answer", "")
    reference = example.outputs.get("answer", "")
    question = example.inputs.get("question", "")

    # Create pre-configured LLM with base_url and api_key
    judge_llm = ChatOpenAI(
        model="gpt-4o",
        temperature=0,
        base_url=base_url,
        api_key=api_key
    )

    # Create judge prompt
    judge_prompt = f"""You are evaluating an AI answer against a reference answer.

Question: {question}
Reference Answer: {reference}
AI Answer: {prediction}

Rate the AI answer on a scale of 0.0 to 1.0:
- 1.0: Perfect match or semantically equivalent
- 0.8: Good, minor differences
- 0.5: Partially correct
- 0.0: Incorrect or unrelated

Return ONLY a number between 0.0 and 1.0."""

    try:
        response = judge_llm.invoke(judge_prompt)
        score = float(response.content.strip())
        score = max(0.0, min(1.0, score))  # Clamp between 0 and 1
    except:
        # Fallback to simple matching if LLM fails
        score = 1.0 if reference.lower() in prediction.lower() else 0.0

    return {"key": "llm_judge", "score": score}

print("✅ Evaluators defined")
print("\n💡 Available evaluators:")
print("   1. accuracy_evaluator - Simple string matching")
print("   2. llm_judge_evaluator - LLM-based quality assessment")

In [ ]:
# Run A/B test
from langsmith.evaluation import evaluate

print("🔄 Running A/B test: GPT-4o vs GPT-4o-mini...")

# Evaluate Variant A (GPT-4o)
results_a = evaluate(
    predict_variant_a,
    data=dataset_name,
    evaluators=[accuracy_evaluator],
    experiment_prefix="agent-ab-gpt4o",
    metadata={"variant": "A", "model": "gpt-4o", "test_type": "model_comparison"}
)

# Evaluate Variant B (GPT-4o-mini)
results_b = evaluate(
    predict_variant_b,
    data=dataset_name,
    evaluators=[accuracy_evaluator],
    experiment_prefix="agent-ab-gpt4o-mini",
    metadata={"variant": "B", "model": "gpt-4o-mini", "test_type": "model_comparison"}
)

print(f"\n✅ Variant A (GPT-4o): {results_a.experiment_name}")
print(f"✅ Variant B (GPT-4o-mini): {results_b.experiment_name}")

In [ ]:
# Pairwise comparison
from langsmith.evaluation import evaluate_comparative
from typing import Sequence, Optional

def model_comparison_evaluator(runs: Sequence[Run], example: Optional[Example] = None) -> dict:
    """Compare two model outputs based on accuracy"""
    output_a = runs[0].outputs.get("answer", "").lower()
    output_b = runs[1].outputs.get("answer", "").lower()
    reference = example.outputs.get("answer", "").lower() if example else ""

    # Score based on reference match
    score_a = 1.0 if reference in output_a else 0.0
    score_b = 1.0 if reference in output_b else 0.0

    return {
        "key": "model_comparison",
        "scores": {str(runs[0].id): score_a, str(runs[1].id): score_b},
        "comment": f"A={score_a}, B={score_b}"
    }

# Run pairwise comparison
comparison = evaluate_comparative(
    (results_a.experiment_name, results_b.experiment_name),
    evaluators=[model_comparison_evaluator],
    experiment_prefix="agent-ab-comparison",
    metadata={"test": "GPT-4o vs GPT-4o-mini"}
)

print("✅ Pairwise comparison complete!")

## Test 2: Prompt Engineering Variations

In [ ]:
# Variant A: Detailed prompt with step-by-step instructions
detailed_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a math tutor. Follow these steps:
    1. Read the problem carefully
    2. Identify what is being asked
    3. Extract relevant numbers and operations
    4. Solve step by step
    5. Provide the final answer with units

    Show your work clearly."""),
    ("user", "{question}")
])

# Variant B: Concise prompt
concise_prompt = ChatPromptTemplate.from_messages([
    ("system", "Solve the math problem. Show your work and provide the final answer."),
    ("user", "{question}")
])

@traceable(name="agent_detailed_prompt")
def agent_detailed_prompt(question: str) -> dict:
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, base_url=base_url, api_key=api_key)
    chain = detailed_prompt | llm | StrOutputParser()
    return {"answer": chain.invoke({"question": question}), "prompt": "detailed"}

@traceable(name="agent_concise_prompt")
def agent_concise_prompt(question: str) -> dict:
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, base_url=base_url, api_key=api_key)
    chain = concise_prompt | llm | StrOutputParser()
    return {"answer": chain.invoke({"question": question}), "prompt": "concise"}

print("✅ Prompt variants defined")
print("\n💡 Test Hypothesis:")
print("   Detailed prompts may improve accuracy but increase latency and token usage")

## Test 3: Temperature Tuning

In [ ]:
# Test different temperature settings
@traceable(name="agent_temp_0")
def agent_temp_0(question: str) -> dict:
    """Deterministic output (temperature=0)"""
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, base_url=base_url, api_key=api_key)
    chain = agent_prompt | llm | StrOutputParser()
    return {"answer": chain.invoke({"question": question}), "temperature": 0}

@traceable(name="agent_temp_0.3")
def agent_temp_03(question: str) -> dict:
    """Slightly creative (temperature=0.3)"""
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3, base_url=base_url, api_key=api_key)
    chain = agent_prompt | llm | StrOutputParser()
    return {"answer": chain.invoke({"question": question}), "temperature": 0.3}

@traceable(name="agent_temp_0.7")
def agent_temp_07(question: str) -> dict:
    """More creative (temperature=0.7)"""
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7, base_url=base_url, api_key=api_key)
    chain = agent_prompt | llm | StrOutputParser()
    return {"answer": chain.invoke({"question": question}), "temperature": 0.7}

print("✅ Temperature variants defined")
print("\n💡 Test Hypothesis:")
print("   Lower temperature → More consistent, deterministic")
print("   Higher temperature → More creative, but less consistent")

# RAG A/B Testing

## Popular RAG A/B Tests:

| Test Type | Variant A | Variant B | What to Measure |

|-----------|-----------|-----------|------------------|

| **Chunk Size** | 500 tokens | 1000 tokens | Precision vs Recall |

| **Chunk Overlap** | 0 tokens | 100 tokens | Context continuity |

| **Retrieval Method** | Similarity | MMR (diversity) | Answer quality |

| **Top-K** | k=3 | k=5 | Precision vs Coverage |

| **Reranking** | No reranking | With reranking | Relevance |

| **Embedding Model** | OpenAI | Custom model | Domain specificity |

| **Query Transform** | Original query | Expanded query | Retrieval quality |

## Test 1: Chunk Size Optimization

In [ ]:
# Fetch sample document
from langchain_community.document_loaders import WikipediaLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
import sys
sys.path.append('/Users/vinotganesan/Learning/LLM & AGENTS/RAG')
from ml_server_embedding import get_embeddings

# Load document
loader = WikipediaLoader(query="Artificial Intelligence", load_max_docs=1)
docs = loader.load()
print(f"✅ Loaded document: {docs[0].metadata['title']} ({len(docs[0].page_content)} chars)")

In [ ]:
# Variant A: Small chunks (500 tokens)
splitter_500 = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks_500 = splitter_500.split_documents(docs)

vectorstore_500 = Chroma.from_documents(
    documents=chunks_500,
    embedding=get_embeddings(),
    collection_name="rag_ab_500"
)
retriever_500 = vectorstore_500.as_retriever(search_kwargs={"k": 3})

print(f"✅ Variant A: {len(chunks_500)} chunks (size=500)")

In [ ]:
# Variant B: Large chunks (1000 tokens)
splitter_1000 = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
chunks_1000 = splitter_1000.split_documents(docs)

vectorstore_1000 = Chroma.from_documents(
    documents=chunks_1000,
    embedding=get_embeddings(),
    collection_name="rag_ab_1000"
)
retriever_1000 = vectorstore_1000.as_retriever(search_kwargs={"k": 3})

print(f"✅ Variant B: {len(chunks_1000)} chunks (size=1000)")

In [ ]:
# RAG chains
import openai
from langsmith.wrappers import wrap_openai

openai_client = wrap_openai(openai.Client(base_url=base_url, api_key=api_key))

@traceable(name="rag_chunk_500")
def rag_chunk_500(question: str) -> dict:
    """RAG with 500-token chunks"""
    docs = retriever_500.invoke(question)
    context = "\n\n".join([doc.page_content for doc in docs])

    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": f"Answer based on this context:\n{context}"},
            {"role": "user", "content": question}
        ],
        temperature=0
    )

    return {"answer": response.choices[0].message.content, "chunk_size": 500, "contexts": [doc.page_content for doc in docs]}

@traceable(name="rag_chunk_1000")
def rag_chunk_1000(question: str) -> dict:
    """RAG with 1000-token chunks"""
    docs = retriever_1000.invoke(question)
    context = "\n\n".join([doc.page_content for doc in docs])

    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": f"Answer based on this context:\n{context}"},
            {"role": "user", "content": question}
        ],
        temperature=0
    )

    return {"answer": response.choices[0].message.content, "chunk_size": 1000, "contexts": [doc.page_content for doc in docs]}

print("✅ RAG variants defined")

In [ ]:
# Create RAG dataset
dataset_name_rag = "AI_QA_RAG"

rag_examples = [
    ("What is artificial intelligence?", "AI is the simulation of human intelligence by machines"),
    ("What are the main types of AI?", "Narrow AI and General AI"),
    ("What is machine learning?", "Machine learning is a subset of AI that learns from data"),
]

try:
    dataset_rag = client.read_dataset(dataset_name=dataset_name_rag)
    print(f"✅ Using existing dataset: {dataset_name_rag}")
except:
    dataset_rag = client.create_dataset(dataset_name=dataset_name_rag, description="AI questions for RAG evaluation")
    inputs_list = [{"question": q} for q, _ in rag_examples]
    outputs_list = [{"answer": a} for _, a in rag_examples]
    client.create_examples(inputs=inputs_list, outputs=outputs_list, dataset_id=dataset_rag.id)
    print(f"✅ Created dataset with {len(rag_examples)} examples")

In [ ]:
# Run chunk size A/B test
def predict_rag_500(example: dict) -> dict:
    return rag_chunk_500(example["question"])

def predict_rag_1000(example: dict) -> dict:
    return rag_chunk_1000(example["question"])

# Evaluate both variants
print("🔄 Running RAG A/B test: Chunk size 500 vs 1000...")

results_rag_500 = evaluate(
    predict_rag_500,
    data=dataset_name_rag,
    evaluators=[accuracy_evaluator],
    experiment_prefix="rag-ab-chunk-500",
    metadata={"variant": "A", "chunk_size": 500, "test_type": "chunk_size"}
)

results_rag_1000 = evaluate(
    predict_rag_1000,
    data=dataset_name_rag,
    evaluators=[accuracy_evaluator],
    experiment_prefix="rag-ab-chunk-1000",
    metadata={"variant": "B", "chunk_size": 1000, "test_type": "chunk_size"}
)

print(f"\n✅ Variant A (chunk=500): {results_rag_500.experiment_name}")
print(f"✅ Variant B (chunk=1000): {results_rag_1000.experiment_name}")

## Test 2: Retrieval Strategy (Similarity vs MMR)

In [ ]:
# Variant A: Similarity search (default)
retriever_similarity = vectorstore_1000.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

# Variant B: MMR (Maximal Marginal Relevance - promotes diversity)
retriever_mmr = vectorstore_1000.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 3, "fetch_k": 10}  # Fetch 10, return 3 diverse
)

@traceable(name="rag_similarity")
def rag_similarity(question: str) -> dict:
    """RAG with similarity search"""
    docs = retriever_similarity.invoke(question)
    context = "\n\n".join([doc.page_content for doc in docs])

    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": f"Answer based on this context:\n{context}"},
            {"role": "user", "content": question}
        ],
        temperature=0
    )

    return {"answer": response.choices[0].message.content, "retrieval": "similarity"}

@traceable(name="rag_mmr")
def rag_mmr(question: str) -> dict:
    """RAG with MMR (diverse) search"""
    docs = retriever_mmr.invoke(question)
    context = "\n\n".join([doc.page_content for doc in docs])

    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": f"Answer based on this context:\n{context}"},
            {"role": "user", "content": question}
        ],
        temperature=0
    )

    return {"answer": response.choices[0].message.content, "retrieval": "mmr"}

print("✅ Retrieval strategy variants defined")
print("\n💡 Test Hypothesis:")
print("   Similarity → More focused, might miss diverse aspects")
print("   MMR → More diverse context, better for broad questions")

## Test 3: Top-K Optimization (k=3 vs k=5)

In [ ]:
# Variant A: Retrieve top-3 documents
retriever_k3 = vectorstore_1000.as_retriever(search_kwargs={"k": 3})

# Variant B: Retrieve top-5 documents
retriever_k5 = vectorstore_1000.as_retriever(search_kwargs={"k": 5})

@traceable(name="rag_k3")
def rag_k3(question: str) -> dict:
    docs = retriever_k3.invoke(question)
    context = "\n\n".join([doc.page_content for doc in docs])

    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": f"Answer based on this context:\n{context}"},
            {"role": "user", "content": question}
        ],
        temperature=0
    )

    return {"answer": response.choices[0].message.content, "k": 3}

@traceable(name="rag_k5")
def rag_k5(question: str) -> dict:
    docs = retriever_k5.invoke(question)
    context = "\n\n".join([doc.page_content for doc in docs])

    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": f"Answer based on this context:\n{context}"},
            {"role": "user", "content": question}
        ],
        temperature=0
    )

    return {"answer": response.choices[0].message.content, "k": 5}

print("✅ Top-K variants defined")
print("\n💡 Test Hypothesis:")
print("   k=3 → More precise, lower cost, faster")
print("   k=5 → Better coverage, higher cost, more context")

# Results Analysis

## Key Metrics to Track:

### Agent Metrics:

- **Accuracy**: Task completion rate

- **Latency**: Response time

- **Cost**: Token usage

- **Consistency**: Variance across runs (use `num_repetitions`)

### RAG Metrics:

- **Faithfulness**: Answer grounded in retrieved context

- **Answer Relevancy**: Relevance to question

- **Context Precision**: Relevant docs ranked higher

- **Context Recall**: Ground truth info in context

## How to Analyze Results:

1. **Go to LangSmith UI**

2. **Navigate to Datasets** → Select your dataset

3. **Click Compare** → Select both experiments

4. **View side-by-side**:

   - Accuracy scores

   - Latency distribution

   - Cost comparison

   - Individual examples

5. **Statistical Significance**:

   - Use `num_repetitions=5+` for robust results

   - Check win rate (% times Variant A > Variant B)

   - Look for consistent patterns

## Decision Framework:

In [ ]:
if accuracy_improvement > 5% and cost_increase < 20%:
    → Choose higher-performing variant

elif cost_reduction > 30% and accuracy_loss < 2%:
    → Choose cost-effective variant

else:
    → Run more tests or hybrid approach

In [ ]:
# Summary of all A/B tests
print("="*70)
print("A/B TESTING SUMMARY")
print("="*70)

print("\n📊 AGENT TESTS:")
print("   1. Model Comparison: GPT-4o vs GPT-4o-mini")
print("   2. Prompt Engineering: Detailed vs Concise")
print("   3. Temperature: 0 vs 0.3 vs 0.7")

print("\n📊 RAG TESTS:")
print("   1. Chunk Size: 500 vs 1000 tokens")
print("   2. Retrieval: Similarity vs MMR")
print("   3. Top-K: k=3 vs k=5")

print("\n💡 NEXT STEPS:")
print("   1. Review results in LangSmith UI")
print("   2. Compare metrics (accuracy, latency, cost)")
print("   3. Choose winning variant or iterate")
print("   4. Deploy to production")
print("="*70)

## Additional A/B Test Ideas

### Agent Tests:

1. **ReAct vs Plan-and-Execute** - Different agent frameworks

2. **Tool Selection** - All tools vs curated subset

3. **Context Window** - Full context vs summarized

4. **Multi-step vs Single-step** - Agent complexity

5. **With/without memory** - Stateful vs stateless agents

### RAG Tests:

1. **Embedding Models** - OpenAI vs Cohere vs HuggingFace

2. **Reranking** - With vs without Cohere reranker

3. **Query Transformation** - Original vs HyDE vs Multi-query

4. **Hybrid Search** - Semantic + Keyword vs Semantic only

5. **Chunking Strategy** - Recursive vs Semantic vs Fixed-size

6. **Parent-Child Retrieval** - Retrieve small, return large chunks

### Combined Tests:

1. **Agent + RAG** - Different agent models with different RAG configs

2. **Cost Optimization** - Quality vs Cost tradeoffs

3. **Latency Optimization** - Speed vs Accuracy tradeoffs